## Step 1 — download roads layer from osmnx

**# of cells in notebook:** 1

**Purpose:** Download drivable OSM road network over a user defined area using the osmnx package. 

**Input:**

- a geodatabase with a blocks layer for the city, provided by GRID3 team. Input blocks must be in WGS84.

**Output:**

- `extent_hull`, `roads_1`, layers in the same geodatabase as the input data.

**Main logic:**

1. dissolve blocks to get city extent
2. create convex hull over a singlepart city extent
3. use convex hull to extract roads data
4. write roads to geodatabase as roads_1

In [ ]:
import os
import time
import warnings

import geopandas as gpd
import osmnx as ox
import pandas as pd
import pyogrio


# -------------------------------------------------------------------
# INPUTS / OUTPUTS
# -------------------------------------------------------------------
input_gdb = r"E:\World Bank deliverbale 1\_analysis\blocks\blocks.gdb"
input_layer = "juba_blocks_20260415_reordered"

out_gdb = r"E:\World Bank deliverbale 1\_analysis\roads\roads.gdb"

extent_hull_layer = "extent_hull"
out_layer = "roads_1"


# -------------------------------------------------------------------
# OSMNX SETTINGS
# -------------------------------------------------------------------
ox.settings.use_cache = True
ox.settings.log_console = True
ox.settings.requests_timeout = 180


# -------------------------------------------------------------------
# HELPER FUNCTIONS
# -------------------------------------------------------------------
def print_gdf_info(gdf, name):
    print(f"\n{name}")
    print(f"  Rows: {len(gdf):,}")
    print(f"  CRS: {gdf.crs}")
    print(f"  Geometry types: {sorted(gdf.geometry.geom_type.unique())}")
    print(f"  Bounds: {gdf.total_bounds}")


def clean_value(x):
    """
    Convert list/dict-like OSMnx attribute values to strings so that
    file geodatabase output can handle them reliably.
    """
    if x is None:
        return None

    try:
        if pd.isna(x):
            return None
    except Exception:
        pass

    if isinstance(x, (list, tuple, set)):
        return "; ".join(str(v) for v in x)

    if isinstance(x, dict):
        return "; ".join(f"{k}: {v}" for k, v in x.items())

    return x


def make_safe_field_name(name, used):
    """
    Make field names safer for file geodatabase output.
    """
    safe = (
        str(name)
        .replace(":", "_")
        .replace("-", "_")
        .replace(" ", "_")
        .replace(".", "_")
        .replace("/", "_")
        .replace("\\", "_")
        .replace("(", "_")
        .replace(")", "_")
    )

    if not safe:
        safe = "field"

    if safe[0].isdigit():
        safe = f"f_{safe}"

    safe = safe[:60]

    reserved = {
        "objectid",
        "fid",
        "shape",
        "shape_length",
        "shape_area",
        "geometry",
    }

    base = safe
    i = 1

    while safe.lower() in used or safe.lower() in reserved:
        suffix = f"_{i}"
        safe = f"{base[:60 - len(suffix)]}{suffix}"
        i += 1

    used.add(safe.lower())
    return safe


def clean_edges_for_output(edges_gdf):
    """
    Prepare OSMnx edge GeoDataFrame for FileGDB writing.
    """
    edges_gdf = edges_gdf.copy()

    # Bring u, v, key into regular columns.
    if isinstance(edges_gdf.index, pd.MultiIndex):
        edges_gdf = edges_gdf.reset_index()
    else:
        edges_gdf = edges_gdf.reset_index(drop=False)

    geom_col = edges_gdf.geometry.name

    # Convert list/dict/object values to scalar-friendly strings.
    for col in edges_gdf.columns:
        if col == geom_col:
            continue

        if edges_gdf[col].dtype == "object":
            edges_gdf[col] = edges_gdf[col].apply(clean_value)

    # Rename fields safely.
    used = set()
    rename = {}

    for col in edges_gdf.columns:
        if col == geom_col:
            continue

        rename[col] = make_safe_field_name(col, used)

    edges_gdf = edges_gdf.rename(columns=rename)

    return edges_gdf


def write_to_filegdb(gdf, gdb_path, layer_name):
    """
    Write a GeoDataFrame directly to a FileGDB using OpenFileGDB.

    Important:
    This intentionally does NOT call pyogrio.list_layers(), because
    list_layers() previously failed in this environment even though
    direct writing worked.
    """
    if not os.path.exists(gdb_path):
        raise FileNotFoundError(f"Output geodatabase does not exist: {gdb_path}")

    if not os.path.isdir(gdb_path):
        raise NotADirectoryError(f"Output geodatabase path is not a directory: {gdb_path}")

    print("\n--- Writing output to FileGDB ---")
    print(f"GDB:   {gdb_path}")
    print(f"Layer: {layer_name}")
    print("Skipping pre-check for existing layer.")
    print("Make sure this layer name does not already exist in the geodatabase.")

    pyogrio.write_dataframe(
        gdf,
        gdb_path,
        layer=layer_name,
        driver="OpenFileGDB",
        layer_options={
            "TARGET_ARCGIS_VERSION": "ARCGIS_PRO_3_2_OR_LATER"
        }
    )

    print("Successfully wrote FileGDB layer.")
    print(f"Output: {gdb_path}\\{layer_name}")


# -------------------------------------------------------------------
# RUN
# -------------------------------------------------------------------
start_time = time.time()

print("Starting OSMnx Juba drivable road download...")
print(f"Input GDB:        {input_gdb}")
print(f"Input layer:      {input_layer}")
print(f"Output GDB:       {out_gdb}")
print(f"Extent hull layer:{extent_hull_layer}")
print(f"Roads layer:      {out_layer}")

# -------------------------------------------------------------------
# 0. Check output geodatabase path and driver
# -------------------------------------------------------------------
print("\n--- Step 0: Check output geodatabase ---")

if not os.path.exists(out_gdb):
    raise FileNotFoundError(f"Output geodatabase does not exist: {out_gdb}")

if not os.path.isdir(out_gdb):
    raise NotADirectoryError(f"Output geodatabase path is not a directory: {out_gdb}")

print("OpenFileGDB driver status:", pyogrio.list_drivers().get("OpenFileGDB"))

print(
    "Skipping pyogrio.list_layers() because it previously failed on this "
    "geodatabase/environment, even though direct writing works."
)

# -------------------------------------------------------------------
# 1. Read blocks layer
# -------------------------------------------------------------------
print("\n--- Step 1: Read blocks layer ---")

blocks = gpd.read_file(input_gdb, layer=input_layer)

if blocks.empty:
    raise ValueError("Input blocks layer is empty.")

print_gdf_info(blocks, "Input blocks")

# Input is expected to be WGS84.
if blocks.crs is None:
    warnings.warn(
        "Input CRS is missing. Setting CRS to EPSG:4326 based on your description."
    )
    blocks = blocks.set_crs(epsg=4326)
elif blocks.crs.to_epsg() != 4326:
    print("Input blocks are not EPSG:4326. Reprojecting to WGS84.")
    blocks = blocks.to_crs(epsg=4326)

# -------------------------------------------------------------------
# 2. Repair invalid input geometries if needed
# -------------------------------------------------------------------
print("\n--- Step 2: Check and repair input geometries ---")

invalid_count = int((~blocks.geometry.is_valid).sum())
print(f"Invalid input geometries: {invalid_count}")

if invalid_count > 0:
    print("Repairing invalid geometries using buffer(0).")
    blocks["geometry"] = blocks.geometry.buffer(0)

# -------------------------------------------------------------------
# 3. Dissolve blocks to one single multipart feature
# -------------------------------------------------------------------
print("\n--- Step 3: Dissolve blocks to one multipart feature ---")

blocks_dissolved = blocks.dissolve()

if blocks_dissolved.empty:
    raise ValueError("Dissolved blocks output is empty.")

dissolved_geom = blocks_dissolved.geometry.iloc[0]

if not dissolved_geom.is_valid:
    print("Dissolved geometry is invalid. Repairing with buffer(0).")
    dissolved_geom = dissolved_geom.buffer(0)

blocks_dissolved = gpd.GeoDataFrame(
    {"name": ["juba_blocks_dissolved"]},
    geometry=[dissolved_geom],
    crs="EPSG:4326"
)

print_gdf_info(blocks_dissolved, "Dissolved blocks multipart feature")
print(f"Dissolved geometry type: {dissolved_geom.geom_type}")

# -------------------------------------------------------------------
# 4. Create convex hull from the dissolved multipart feature
# -------------------------------------------------------------------
print("\n--- Step 4: Create convex hull from dissolved blocks ---")

convex_hull_geom = dissolved_geom.convex_hull

if not convex_hull_geom.is_valid:
    print("Convex hull is invalid. Repairing with buffer(0).")
    convex_hull_geom = convex_hull_geom.buffer(0)

extent_hull_gdf = gpd.GeoDataFrame(
    {
        "name": ["juba_blocks_convex_hull"],
        "source": ["juba_blocks_20260415_reordered"],
        "method": ["dissolve_then_convex_hull"]
    },
    geometry=[convex_hull_geom],
    crs="EPSG:4326"
)

print_gdf_info(extent_hull_gdf, "Convex hull download polygon")
print(f"Convex hull geometry type: {convex_hull_geom.geom_type}")
print(f"Convex hull bounds: {convex_hull_geom.bounds}")

# -------------------------------------------------------------------
# 5. Write convex hull to geodatabase as extent_hull
# -------------------------------------------------------------------
print("\n--- Step 5: Write convex hull to geodatabase ---")

write_to_filegdb(
    extent_hull_gdf,
    out_gdb,
    extent_hull_layer
)

# -------------------------------------------------------------------
# 6. Download drivable OSM network using convex hull
# -------------------------------------------------------------------
print("\n--- Step 6: Download OSM drivable network using convex hull ---")

G = ox.graph_from_polygon(
    polygon=convex_hull_geom,
    network_type="drive",
    simplify=False,
    retain_all=True,
    truncate_by_edge=True
)

print("Downloaded graph:")
print(f"  Nodes: {len(G.nodes):,}")
print(f"  Edges: {len(G.edges):,}")

if len(G.edges) == 0:
    raise ValueError(
        "OSMnx returned zero edges. Check the convex hull polygon and Overpass availability."
    )

# -------------------------------------------------------------------
# 7. Convert graph to edges-only GeoDataFrame
# -------------------------------------------------------------------
print("\n--- Step 7: Convert graph to edges-only GeoDataFrame ---")

edges = ox.graph_to_gdfs(
    G,
    nodes=False,
    edges=True,
    node_geometry=False,
    fill_edge_geometry=True
)

if edges.empty:
    raise ValueError("Edges GeoDataFrame is empty.")

if edges.crs is None:
    edges = edges.set_crs(epsg=4326)
elif edges.crs.to_epsg() != 4326:
    edges = edges.to_crs(epsg=4326)

print_gdf_info(edges, "Downloaded edges")

# -------------------------------------------------------------------
# 8. Clean fields for FileGDB output
# -------------------------------------------------------------------
print("\n--- Step 8: Clean edge attributes for FileGDB output ---")

edges_clean = clean_edges_for_output(edges)

print_gdf_info(edges_clean, "Cleaned edges")

print("\nOutput fields:")
for col in edges_clean.columns:
    print(f"  {col}")

# -------------------------------------------------------------------
# 9. Write roads to geodatabase as roads_1
# -------------------------------------------------------------------
print("\n--- Step 9: Write roads to geodatabase ---")

write_to_filegdb(
    edges_clean,
    out_gdb,
    out_layer
)

elapsed = time.time() - start_time
print(f"\nDone. Processing completed in {elapsed / 60:.2f} minutes.")